In [ ]:
from __future__ import print_function

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np
import random
import argparse
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, roc_auc_score
import os
import collections
from torch.utils.data import DataLoader, Subset
import torch.nn.functional as F
from tabulate import tabulate
import time
import matplotlib.pyplot as plt
import csv
import pickle

device = torch.device("cpu")

# Define the CNN model (LeNet)
class CNN(nn.Module):


    def __init__(self):
        super(CNN, self).__init__()
        # 1 input image channel, 6 output channels, 3x3 square convolution
        hidden_channels = 24
        self.layer1a = nn.Sequential(
            nn.Conv2d(in_channels=1,
                      out_channels=hidden_channels,
                      kernel_size=3, padding=1),
            #nn.BatchNorm2d(hidden_channels),
            nn.ReLU()
        )
        # H=(28+2-3)/1+1=28
        self.layer1b = nn.Sequential(
            nn.Conv2d(in_channels=hidden_channels,
                      out_channels=hidden_channels,
                      kernel_size=3, padding=1),
            #nn.BatchNorm2d(hidden_channels),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        # H=(28+2-3)/1+1=28
        # H=(28-2)/2+1=14
        # output shape=[batch_size, hidden_channels, 14, 14]

        self.layer2a = nn.Sequential(
            nn.Conv2d(in_channels=hidden_channels,
                      out_channels=hidden_channels*2,
                      kernel_size=3, padding=1),
            #nn.BatchNorm2d(hidden_channels*2),
            nn.ReLU(),
        )
        # H=(14+2-3)/1+1=14
        self.layer2b = nn.Sequential(
            nn.Conv2d(in_channels=hidden_channels*2,
                      out_channels=hidden_channels*2,
                      kernel_size=3, padding=0),
            #nn.BatchNorm2d(hidden_channels*2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        # H=(14+0-3)/1+1=12
        # H=(12-2)/2+1=6
        # output shape=[batch_size, hidden_channels*2, 6, 6]

        self.layer3a = nn.Sequential(
            nn.Conv2d(in_channels=hidden_channels*2,
                      out_channels=hidden_channels*4,
                      kernel_size=3, padding=1),
            #nn.BatchNorm2d(hidden_channels*4),
            nn.ReLU(),
        )
        # H=(6+2-3)/1+1=6
        self.layer3b = nn.Sequential(
            nn.Conv2d(in_channels=hidden_channels*4,
                      out_channels=hidden_channels*4,
                      kernel_size=3, padding=0),
            #nn.BatchNorm2d(hidden_channels*4),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        # H=(6+0-3)/1+1=4
        # H=(4-2)/2 + 1=2
        # output shape=[batch_size, hidden_channels*4, 2, 2]

        """No need dropout, ReLU, BatchNorm for this model after doing grid-search
        One layer is enough.
        """
        #fcs = [hidden_channels*4*2*2, 120, 84, 10]
        #fcs = [hidden_channels*4*2*2, 96, 10]
        fcs = [hidden_channels*4*2*2, 10]
        self.fc1 = nn.Linear(in_features=fcs[0],out_features=fcs[1])
        #self.fc2 = nn.Linear(in_features=fcs[1], out_features=fcs[2])
        #self.fc3 = nn.Linear(in_features=fcs[2], out_features=fcs[3])

    def forward(self, x):
        out = self.layer1a(x)
        out = self.layer1b(out)
        out = self.layer2a(out)
        out = self.layer2b(out)
        out = self.layer3a(out)
        out = self.layer3b(out)
        #out = out.view(out.size(0), -1)
        out = torch.flatten(out, start_dim=1)
        out = self.fc1(out)
        #out = self.fc2(out)
        #out = self.fc3(out)
        return out

# Evaluation metrics functions
def calculate_ter(model, test_loader, device):
    model.eval()
    total = 0
    incorrect = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            _, predicted = torch.max(output.data, 1)
            total += target.size(0)
            incorrect += (predicted != target).sum().item()
    return incorrect / total * 100  # Return as percentage

def calculate_asr(model, test_loader, target_label, trigger_fn, device):
    model.eval()
    total = 0
    success = 0
    with torch.no_grad():
        for data, target in test_loader:
            mask = (target != target_label)
            data, target = data[mask], target[mask]
            if len(data) == 0:
                continue
            triggered_data = trigger_fn(data)
            output = model(triggered_data.to(device))
            _, predicted = torch.max(output.data, 1)
            total += target.size(0)
            success += (predicted == target_label).sum().item()
    return success / total * 100 if total > 0 else 0  # Return as percentage

def calculate_acp(T, Tr_list):
    cp_list = [(T - Tr) / T * 100 for Tr in Tr_list]
    return sum(cp_list) / len(cp_list) if cp_list else 0

# Aggregation functions
def simple_mean(old_gradients, param_list, net, lr, b=0, hvp=None, device='cpu'):
    if hvp is not None:
        pred_grad = []
        distance = []
        for i in range(len(old_gradients)):
            pred_grad.append(old_gradients[i] + hvp[i])
        old_concat = torch.cat(old_gradients, dim=1)
        param_concat = torch.cat(param_list, dim=1)
        distance = torch.norm(old_concat - param_concat, dim=0).cpu().numpy()
        auc1 = roc_auc_score(np.concatenate([np.ones(b), np.zeros(len(hvp) - b)]), distance)
        pred_concat = torch.cat(pred_grad, dim=1)
        distance = torch.norm(pred_concat - param_concat, dim=0).cpu().numpy()
        auc2 = roc_auc_score(np.concatenate([np.ones(b), np.zeros(len(hvp) - b)]), distance)
        print(f"Detection AUC: {auc1:.4f}; Detection AUC: {auc2:.4f}")
        distance = distance / np.sum(distance)
    else:
        distance = None

    mean_tensor = torch.mean(torch.cat(param_list, dim=1), dim=-1, keepdim=True)
    idx = 0
    with torch.no_grad():
        for param in net.parameters():
            if param.requires_grad:
                param_size = param.numel()
                param.data -= lr * mean_tensor[idx:(idx + param_size)].view(param.shape)
                idx += param_size
    return mean_tensor, distance

# Byzantine attack functions
def no_byz(v, f):
    return v

def gaussian_attack(v, f, epsilon=0.01):
    vi_shape = v[0].shape
    adv_grads = torch.cat(v, dim=1)
    e_mu = torch.mean(adv_grads, dim=1)
    e_sigma = torch.sqrt(torch.sum((adv_grads - e_mu.unsqueeze(1)) ** 2, dim=1) / f)
    for i in range(f):
        norm = torch.norm(v[i])
        v[i] = torch.normal(e_mu, e_sigma).view(vi_shape)
        v[i] *= norm / torch.norm(v[i])
    return v

# Detection functions
def detection1(score, nobyz, num_workers):
    nrefs = 10
    ks = range(1, 8)
    gaps = np.zeros(len(ks))
    gapDiff = np.zeros(len(ks) - 1)
    sdk = np.zeros(len(ks))
    min_score = np.min(score)
    max_score = np.max(score)
    score = (score - min_score) / (max_score - min_score)
    for i, k in enumerate(ks):
        estimator = KMeans(n_clusters=k)
        estimator.fit(score.reshape(-1, 1))
        label_pred = estimator.labels_
        center = estimator.cluster_centers_
        Wk = np.sum([np.square(score[m] - center[label_pred[m]]) for m in range(len(score))])
        WkRef = np.zeros(nrefs)
        for j in range(nrefs):
            rand = np.random.uniform(0, 1, len(score))
            estimator = KMeans(n_clusters=k)
            estimator.fit(rand.reshape(-1, 1))
            label_pred = estimator.labels_
            center = estimator.cluster_centers_
            WkRef[j] = np.sum([np.square(rand[m] - center[label_pred[m]]) for m in range(len(rand))])
        gaps[i] = np.log(np.mean(WkRef)) - np.log(Wk)
        sdk[i] = np.sqrt((1.0 + nrefs) / nrefs) * np.std(np.log(WkRef))
        if i > 0:
            gapDiff[i - 1] = gaps[i - 1] - gaps[i] + sdk[i]
    select_k = 1
    for i in range(len(gapDiff)):
        if gapDiff[i] >= 0:
            select_k = i + 1
            break
    if select_k == 1:
        print('No attack detected!')
        return 0
    else:
        print('Attack Detected!')
                # Re-run KMeans with the selected number of clusters to get labels
        final_kmeans = KMeans(n_clusters=select_k, n_init=10)
        final_kmeans.fit(score.reshape(-1, 1))
        labels = final_kmeans.labels_
        centers = final_kmeans.cluster_centers_

        # Identify the malicious cluster (assumed to have the highest mean score)
        malicious_cluster = np.argmax(centers)

        # Return indices of malicious clients
        malicious_clients = [i for i in range(len(score)) if labels[i] == malicious_cluster]

        print(f"Malicious Clients Indices: {malicious_clients}")
        return malicious_clients

def detection(score, nobyz, num_workers):
    estimator = KMeans(n_clusters=2)
    estimator.fit(score.reshape(-1, 1))
    label_pred = estimator.labels_
    if np.mean(score[label_pred == 0]) < np.mean(score[label_pred == 1]):
        label_pred = 1 - label_pred
    real_label = np.ones(num_workers)
    real_label[:nobyz] = 0
    acc = len(label_pred[label_pred == real_label]) / num_workers
    recall = 1 - np.sum(label_pred[:nobyz]) / nobyz
    fpr = 1 - np.sum(label_pred[nobyz:]) / (num_workers - nobyz)
    fnr = np.sum(label_pred[:nobyz]) / nobyz
    print(f"acc {acc:.4f}; recall {recall:.4f}; fpr {fpr:.4f}; fnr {fnr:.4f};")
    silhouette = silhouette_score(score.reshape(-1, 1), label_pred)
    return {'ACC': acc, 'FPR': fpr, 'FNR': fnr, 'silhouette': silhouette}

# Utility functions
def lbfgs(args, delta_W, delta_G, v):
    delta_W = torch.cat(delta_W, dim=1)
    delta_G = torch.cat(delta_G, dim=1)
    A = torch.matmul(delta_W.T, delta_G)
    D = torch.diag(torch.diagonal(A))
    L = torch.tril(A)
    sigma = torch.matmul(delta_G[:, -1].T, delta_W[:, -1]) / torch.matmul(delta_W[:, -1].T, delta_W[:, -1])
    upper_mat = torch.cat([-D, L.T], dim=1)
    lower_mat = torch.cat([L, sigma * torch.matmul(delta_W.T, delta_W)], dim=1)
    mat = torch.cat([upper_mat, lower_mat], dim=0)
    rhs = torch.cat([torch.matmul(delta_G.T, v), sigma * torch.matmul(delta_W.T, v)], dim=0)
    mat_inv = torch.inverse(mat)
    p = torch.matmul(mat_inv, rhs)
    Hv_approx = sigma * v - torch.matmul(torch.cat([delta_G, sigma * delta_W], dim=1), p)
    return Hv_approx

def train_from_scratch(num_rounds, learning_rate, train_loader):
    model = CNN().to(device)
    optimizer = optim.SGD(model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()
    for round in range(num_rounds):
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
    return model
def ExactTraining(each_worker_data, each_worker_label, net,train_loader,Cr, learning_rate, aggregation_rule):
    scratch_model = net
    optimizer = optim.SGD(scratch_model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(1):
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            outputs = scratch_model(data)
            loss = criterion(outputs, target)
            loss.backward()
            optimizer.step()
    return scratch_model

def train_from_scratch_for_window(window_data, lr):
    model = CNN().to(device)
    optimizer = optim.SGD(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    for epoch in range(10):
        for data_loader in window_data:
            for inputs, labels in data_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
    return model

def get_window_data(each_worker_data, each_worker_label, window_size, current_epoch, batch_size=64):
    window_data = []
    for i in range(len(each_worker_data)):
        start_idx = max(0, len(each_worker_data[i]) - window_size)
        data = each_worker_data[i][start_idx:]
        labels = each_worker_label[i][start_idx:]
        dataset = torch.utils.data.TensorDataset(data, labels)
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
        window_data.append(loader)
    return window_data


def evaluate_model(model, data_loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data, target in data_loader:
            data, target = data.to(device), target.to(device)
            outputs = model(data)
            _, predicted = torch.max(outputs.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
    return correct / total * 100

def save_model(model, path):
    torch.save(model.state_dict(), path)

def load_model(model, path, device):
    model.load_state_dict(torch.load(path))
    return model.to(device)
# Modified FedRecover function
def FedRecover(Cr, w_bar, g_bar, learning_rate, Tw, Tc, Tf, tau, aggregation_rule,
              client_datasets, each_worker_data, each_worker_label,  e,recover_accuracy_list,net,train_loader, args,test_loader,
              prev_recovered_model=None):
    device = torch.device("cpu")
    Tr_list = []

        # Use previous recovered model if available
    # if prev_recovered_model is not None:
    #     net.load_state_dict(prev_recovered_model.state_dict())
    w_hat = [torch.cat([param.flatten() for param in net.parameters()]).to(device)]
    recovery_accuracies = []
    recovery_iteration_types = []  # Track type of each iteration ('warmup', 'exact', 'lbfgs')

    # Helper function to evaluate and record accuracy
    def evaluate_and_record(model, iteration_type, t=None):
        current_acc = evaluate_accuracy(test_loader, model)

        recovery_accuracies.append(current_acc)
        recovery_iteration_types.append(iteration_type)
        if t is not None:
            print(f"Recovery {iteration_type} Iteration {t}: Accuracy = {current_acc:.2f}%")
        else:
            print(f"Recovery {iteration_type} Update: Accuracy = {current_acc:.2f}%")
        return current_acc

    # Initial evaluation
    initial_acc = evaluate_and_record(net, "initial")
    # Warm-up phase
    print("initial accuracy",initial_acc)
    for t in range(e, e + Tw):
        trained_model = ExactTraining(each_worker_data, each_worker_label, net,train_loader,Cr, learning_rate, aggregation_rule)
        w_hat.append(torch.cat([param.flatten() for param in trained_model.parameters()]).to(device))
        Tr_list.append(t)
        net= trained_model.to(device)

        # Evaluate at each iteration

        acc=evaluate_and_record(trained_model, "warmup", t)
        recover_accuracy_list.append(acc)
       # net.load_state_dict(trained_model.state_dict())





    # Main phase
    for t in range(e + Tw, len(w_bar) - Tf):
        Abnormal_flag=0
        if (t - (e + Tw) + 1) % Tc == 0:
            trained_model = ExactTraining(each_worker_data, each_worker_label, net,train_loader,Cr,learning_rate, aggregation_rule,)
            w_hat.append(torch.cat([param.flatten() for param in trained_model.parameters()]).to(device))
            Tr_list.append(t)
            acc=evaluate_and_record(trained_model, "exact", t)
            recover_accuracy_list.append(acc)
            net= trained_model.to(device)
            #net.load_state_dict(trained_model.state_dict())
        else:
            if len(w_bar) > 0 and len(g_bar) > 0:
                g_hat = []
                for i in Cr:
                    w_hat_tensor = w_hat[-1]
                    w_bar_tensor = w_bar[t]
                    net=trained_model.to(device)
                    model=net
                    w_hat_state_dict = model.state_dict()
                    offset = 0
                    for key, param in w_hat_state_dict.items():
                        param_flat_size = param.numel()
                        w_hat_state_dict[key] = w_hat_tensor[offset:offset + param_flat_size].reshape(param.shape)
                        offset += param_flat_size
                    w_bar_state_dict = model.state_dict()
                    offset = 0
                    for key, param in w_bar_state_dict.items():
                        param_flat_size = param.numel()
                        w_bar_state_dict[key] = w_bar_tensor[offset:offset + param_flat_size].reshape(param.shape)
                        offset += param_flat_size
                    diff_state_dict = {key: w_hat_state_dict[key] - w_bar_state_dict[key] for key in w_hat_state_dict}
                    diff_params = [param.flatten() for param in diff_state_dict.values()]
                    diff = torch.cat(diff_params) if len(diff_params) > 1 else diff_params[0] if diff_params else torch.tensor([])
                    hvpi = lbfgs(args, [w_bar_tensor], [g_bar[t][i]], diff.reshape(1, -1).T)
                    g_bar_dense = g_bar[t][i].to_dense() if g_bar[t][i].is_sparse else g_bar[t][i]
                    g_hat_i = [g_bar_dense + hvpi]
                    g_hat_i_combined = torch.cat(g_hat_i)

                    if torch.norm(g_hat_i_combined, p=float('inf')) > tau:

                        #g_hat_i_combined = torch.cat([param.flatten() for param in trained_model_ab.parameters()]).to(device)

                        print("abnormal")
                        Abnormal_flag=1



                g_hat.append(g_hat_i_combined)

                g_hat = [torch.cat([p.view(-1, 1) for p in grads], dim=0) for grads in g_hat]
                aggregated_grad = aggregation_rule(old_gradients=None, net=net, lr=learning_rate, param_list=g_hat, hvp=None)

                w_hat_state_dict = model.state_dict()
                offset = 0
                for key, param in w_hat_state_dict.items():
                    param_flat_size = param.numel()
                    w_hat_state_dict[key] = w_hat[-1][offset:offset + param_flat_size].reshape(param.shape)
                    offset += param_flat_size
                aggregated_grad_state_dict = model.state_dict()
                offset = 0
                for key, param in aggregated_grad_state_dict.items():
                    param_flat_size = param.numel()
                    w_hat_state_dict[key] = w_hat_tensor[offset:offset + param_flat_size].view(param.shape)
                    offset += param_flat_size
                updated_params_state_dict = {key: w_hat_state_dict[key] - learning_rate * aggregated_grad_state_dict[key] for key in w_hat_state_dict}
                updated_params = torch.cat([param.flatten() for param in updated_params_state_dict.values()])

                w_hat.append(updated_params)
                if Abnormal_flag==1:
                    trained_model_ab = ExactTraining(each_worker_data, each_worker_label, net,train_loader,Cr, learning_rate, aggregation_rule)
                    acc=evaluate_and_record(trained_model_ab, " Recovery abnormal", t)
                    Tr_list.append(t)
                    recover_accuracy_list.append(acc)
                    trained_model=trained_model_ab

                else:
                     acc=evaluate_and_record(trained_model, "abnormal", t)
                     Tr_list.append(t)
                     recover_accuracy_list.append(acc)
                     trained_model=trained_model

            else:
                w_hat.append(w_hat[-1])

    # Final tuning phase
    for t in range(len(w_bar) - Tf, len(w_bar)):
        trained_model = ExactTraining(each_worker_data, each_worker_label, net,train_loader,Cr, learning_rate, aggregation_rule)
        w_hat.append(torch.cat([param.flatten() for param in trained_model.parameters()]).to(device))
        Tr_list.append(t)

        current_acc = evaluate_accuracy(test_loader, trained_model)
        recovery_accuracies.append(current_acc)
        recover_accuracy_list.append(acc)
        print("fine tuning acc",current_acc)
        net= trained_model.to(device)

    # Reconstruct final model
    recovered_model = net.__class__().to(device)
    pointer = 0
    for param in recovered_model.parameters():
        num_elements = param.numel()
        param.data = w_hat[-1][pointer:pointer + num_elements].reshape(param.shape)
        pointer += num_elements

    return recovered_model, Tr_list, recovery_accuracies,recover_accuracy_list

def params_convert(net):
    return torch.cat([p.data.view(-1, 1) for p in net.parameters()], dim=0)

def evaluate_accuracy(data_loader, net):
    net.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for data, target in data_loader:
            data, target = data.to(device), target.to(device)
            outputs = net(data)
            _, predicted = torch.max(outputs, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
    net.train()
    return correct / total * 100

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--dataset", default='mnist', type=str)
    parser.add_argument("--bias", type=float, default=0.1)
    parser.add_argument("--net", default='cnn', type=str, choices=['cnn'])
    parser.add_argument("--batch_size", default=32, type=int)
    parser.add_argument("--lr", default=0.0002, type=float)
    parser.add_argument("--nworkers", default=40, type=int)
    parser.add_argument("--nepochs", default=40, type=int)
    parser.add_argument("--gpu", default=0, type=int)
    parser.add_argument("--seed", default=41, type=int)
    parser.add_argument("--nbyz", default=28, type=int)
    parser.add_argument("--byz_type", default='gaussian', type=str, choices=['no', 'gaussian', 'label_flip'])
    parser.add_argument("--aggregation", default='simple_mean', type=str, choices=['simple_mean'])
    args, _ = parser.parse_known_args()
    return args

def display_detection_metrics(attack_types, detection_methods, metrics_data):
    headers = ["Attack Type", "Detection Method", "Defense ACC", "FPR", "FNR"]
    table_data = []
    for attack in attack_types:
        for method in detection_methods:
            if attack in metrics_data and method in metrics_data[attack]:
                metrics = metrics_data[attack][method]
                row = [attack, method, f"{metrics.get('ACC', 0):.4f}", f"{metrics.get('FPR', 0):.4f}", f"{metrics.get('FNR', 0):.4f}"]
                table_data.append(row)
    print("\nDetection Performance Metrics:")
    print(tabulate(table_data, headers=headers, tablefmt="grid"))
    print("\n")

def load_clients_deltas_trained_on_iteration(path, iteration):
    try:
        deltas_recovered = torch.load(f'{path}/trained_clients_before_aggregation_values_{iteration}.pt', weights_only=False)
        return deltas_recovered
    except FileNotFoundError:
        print(f"File for iteration {iteration} not found, generating dummy data.")
        return [CNN().state_dict() for _ in range(40)]

def load_aggregated_model_on_iteration(path, iteration, device, model):
    try:
        model.to(device)
        model.load_state_dict(torch.load(f'{path}/aggregated_model_{iteration}.pth', weights_only=True))
        model.eval()
        return model
    except FileNotFoundError:
        print(f"Aggregated model for iteration {iteration} not found, returning initialized model.")
        return CNN().to(device)

def main(args):
    torch.manual_seed(args.seed)
    random.seed(args.seed)
    np.random.seed(args.seed)
    num_workers, lr, epochs = args.nworkers, args.lr, args.nepochs
    byz = {'no': no_byz, 'gaussian': gaussian_attack, 'label_flip': no_byz}[args.byz_type]
    agg = {'simple_mean': simple_mean}[args.aggregation]

    transform = transforms.Compose([
        transforms.Resize((32, 32)),
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])
    train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
    test_dataset = torchvision.datasets.MNIST(root='./data', train=False, transform=transform)
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

    each_worker_data, each_worker_label = [[] for _ in range(num_workers)], [[] for _ in range(num_workers)]
    bias_weight, worker_per_group = args.bias, num_workers / 10
    other_group_size = (1 - bias_weight) / 9.
    for data, label in train_loader:
        data, label = data.to(device), label.to(device)
        for x, y in zip(data, label):
            x, y = x.unsqueeze(0), y.unsqueeze(0)
            upper_bound = y.item() * other_group_size + bias_weight
            lower_bound = y.item() * other_group_size
            rd = np.random.random_sample()
            worker_group = (int(np.floor((rd - upper_bound) / other_group_size) + y.item() + 1) if rd > upper_bound else
                           int(np.floor(rd / other_group_size)) if rd < lower_bound else y.item())
            selected_worker = int(worker_group * worker_per_group + np.floor(np.random.random_sample() * worker_per_group))
            each_worker_data[selected_worker].append(x)
            each_worker_label[selected_worker].append(y)
    each_worker_data = [torch.cat(d, dim=0) for d in each_worker_data]
    each_worker_label = [torch.cat(l, dim=0) for l in each_worker_label]
    random_order = np.random.RandomState(seed=args.seed).permutation(num_workers)
    each_worker_data = [each_worker_data[i] for i in random_order]
    each_worker_label = [each_worker_label[i] for i in random_order]

    local_folder = "Historacal data PATH"
    experiment = 'Expriment_NAme'
    aggregation_method = "mudhog"
    path = f"{local_folder}/{experiment}/{aggregation_method}"





    model = CNN()
    aggregated_models = []
    Weight_Diff_Record = []
    param_list_r_mud = []
    gradient_Diff_Record = []
    prev_grad_list = None
    counter=0
    for i in range(40):
        client_models = load_clients_deltas_trained_on_iteration(path, i)
        grad_list = []
        for j in range(len(client_models)):
            temp_model = CNN()
            temp_model.load_state_dict(client_models[j])
            grad_list.append([p.data.clone() for p in temp_model.parameters()])
        current_flattened = [torch.cat([p.view(-1, 1) for p in grads], dim=0) for grads in grad_list]
        param_list_r_mud.append(current_flattened)
        if prev_grad_list is not None:
            iteration_diff = []
            for current_client, prev_client in zip(grad_list, prev_grad_list):
                client_diff = [torch.sub(curr, prev) for curr, prev in zip(current_client, prev_client)]
                iteration_diff.append(client_diff)
            gradient_Diff_Record.append(iteration_diff)
        prev_grad_list = grad_list
    gradient_Diff_Record = [
        [torch.cat([p.view(-1, 1) for p in grads], dim=0) for grads in iteration_diff]
        for iteration_diff in gradient_Diff_Record
    ]
    global_acc_list = []
    for i in range(41):
        model = load_aggregated_model_on_iteration(path, i, device, model)
            # Calculate accuracy on test data
        global_accuracy = evaluate_accuracy(test_loader, model)
        global_acc_list.append(global_accuracy)
        print(f"Global (Poisoned) - Accuracy: {global_accuracy:.4f}")
        aggregated_models.append([p.data.clone() for p in model.parameters()])

        if i > 0:
            diff = [
                torch.stack([current - previous for current, previous in zip(current_params, previous_params)])
                for current_params, previous_params in zip(aggregated_models[i], aggregated_models[i - 1])
            ]
            Weight_Diff_Record.append(diff)
    Weight_Diff_Record = [torch.cat([p.view(-1, 1) for p in grads], dim=0) for grads in Weight_Diff_Record]


    recovery_acc_list = []
    full_scratch_acc_list = []
    recover_accuracy_list=[]
    ter_values = []
    asr_values = []
    acp_values = []
    recovery_time_list = []
    full_scratch_time_list = []


    m_values = [0]
    attack_metrics = {'no': {}, 'gaussian': {}, 'label_flip': {}}

    def trigger_fn(data):
        triggered_data = data.clone()
        triggered_data[:, :, -4:, -4:] = 1.0
        return triggered_data

    target_label = 0
    # Modified training loop in main()
    recovery_acc_history = []  # To store all recovery accuracies
    scratch_acc_history = []    # To store all scratch training accuracies
    prev_recovered_model = None
    for m in m_values:
        args.nbyz = int(args.nworkers * m / 100)

        net = CNN().to(device)
        optimizer = optim.SGD(net.parameters(), lr=lr)
        criterion = nn.CrossEntropyLoss()
        grad_list, old_grad_list, weight_record, grad_record = [], [], [], []
        malicious_score = np.zeros((1, args.nworkers))
        param_list_r = []
        e, flag, start, end = 0, 0, 0, 3
        Cr = range(num_workers)
        Tr_list = []

        while e <len(Weight_Diff_Record):

            print("diff record",len(Weight_Diff_Record))
            print(f"Epoch {e}")
            grad_list = []
            for i in Cr:
                optimizer.zero_grad()
                output = net(each_worker_data[i])
                loss = criterion(output, each_worker_label[i])
                loss.backward()
                grad_list.append([p.grad.clone() for p in net.parameters()])
            param_list = [torch.cat([p.view(-1, 1) for p in grads], dim=0) for grads in grad_list]
            #param_list = byz(param_list, args.nbyz)
            param_list_r.append(param_list)
            old_param_list_r = param_list_r_mud
            weight_record = Weight_Diff_Record
            grad_record = gradient_Diff_Record
           # weight = params_convert(net)




            Hvp = []
            if e > 2:
                for i in Cr:
                    hvp = lbfgs(args, weight_record[0:e], [grad_record[epoch][i] for epoch in range(e)], Weight_Diff_Record[e]) if e > 2 else None
                    Hvp.append(hvp)
            else:
                Hvp = None

            [grad, distance] = simple_mean([old_param_list_r[e][i] for i in Cr], param_list, net, lr, len(Cr), Hvp, device)
            if distance is not None and e % 9 > 2:
                print("check issue",len(malicious_score),len(malicious_score[0]),len(distance))
                malicious_score = np.row_stack((malicious_score, distance))

            #global_accuracy = evaluate_accuracy(test_loader, net)
            #global_acc_list.append(global_accuracy)

            if malicious_score.shape[0] >= 2:
                x = detection1(np.sum(malicious_score[:], axis=0), args.nbyz, len(Cr))

                if x:
                    flag=1
                    print(x)
                    print('Stop at iteration:', e)

                    print("remaining clinet",Cr)
                    metrics = detection(np.sum(malicious_score[:], axis=0), args.nbyz, len(Cr))
                    attack_metrics[args.byz_type]['K-Means'] = {
                        'ACC': metrics['ACC'],
                        'FPR': metrics['FPR'],
                        'FNR': metrics['FNR']
                    }
                    Cr=[item for item in Cr if item not in x]
                    malicious_score = np.zeros((1, len(Cr)))


            print("flg",flag)
            if  (e==9 ): # I set e=9 because the window size is 10. If you change the window size (e.g., to 20), then e=19, since indexing starts at 0.

                for i in range(e-9,e+1):
                    prev_recovered_model = load_aggregated_model_on_iteration(path, i, device, model)
                        # Calculate accuracy on test data
                    global_accuracy = evaluate_accuracy(test_loader, prev_recovered_model)
                    recover_accuracy_list.append(global_accuracy)
                    print("epoch recovery in epoch ",global_accuracy)
                prev_recovered_model=prev_recovered_model
                flag = 0
            elif(e==9 and flag==1):
                scratch_model = CNN().to(device)
                optimizer = optim.SGD(scratch_model.parameters(), lr=lr)
                criterion = nn.CrossEntropyLoss()
                scratch_acc_history=[]
                for epoch in range(10):
                    for data, target in train_loader:
                        data, target = data.to(device), target.to(device)
                        optimizer.zero_grad()
                        outputs = scratch_model(data)
                        loss = criterion(outputs, target)
                        loss.backward()
                        optimizer.step()

                    # Evaluate at each epoch
                    global_accuracy = evaluate_accuracy(test_loader, scratch_model)
                    recover_accuracy_list.append(global_accuracy)
                    print("epoch recovery in epoch ",global_accuracy)
                prev_recovered_model=scratch_model
                flag = 0
            elif ((e+1) % 10 == 0  ):

                # Recovery process
                start_time = time.time()
                w_bar = weight_record[0:e+1]
                g_bar = old_param_list_r[0:e+1]
                Tw, Tc, Tf, tau = 2, 3, 2, 0.1
                client_datasets = [[each_worker_data[i], each_worker_label[i]] for i in Cr]
                if e==9:
                    model=load_aggregated_model_on_iteration(path, 0, device, CNN())
                else:
                    model=prev_recovered_model
                net=model.to(device)
                print("cr",len(Cr))
                print(" before recover_accuracy_list",len(recover_accuracy_list))
                # Pass previous recovered model
                recovered_model, Tr_list, iter_accs,recover_accuracy_list= FedRecover(
                    Cr, w_bar, g_bar, lr, Tw, Tc, Tf, tau, agg,
                    client_datasets, [each_worker_data[i] for i in Cr], [each_worker_label[i] for i in Cr],
                     e - 9,recover_accuracy_list,net,train_loader, args, test_loader,prev_recovered_model
                )
                print(" after recover_accuracy_list",len(recover_accuracy_list))
                recovery_time = time.time() - start_time
                recovery_time_list.append(recovery_time)
                recovery_acc_history.extend(iter_accs)
                prev_recovered_model = recovered_model



                # Full scratch training with iteration-level evaluation
                scratch_iter_accs = []
                start_time = time.time()
                scratch_model = CNN().to(device)
                optimizer = optim.SGD(scratch_model.parameters(), lr=lr)
                criterion = nn.CrossEntropyLoss()
                scratch_acc_history=[]
                for epoch in range(len(old_param_list_r)):
                    for data, target in train_loader:
                        data, target = data.to(device), target.to(device)
                        optimizer.zero_grad()
                        outputs = scratch_model(data)
                        loss = criterion(outputs, target)
                        loss.backward()
                        optimizer.step()

                    # Evaluate at each epoch
                    epoch_acc = evaluate_accuracy(test_loader, scratch_model)
                    scratch_iter_accs.append(epoch_acc)
                    print(f"Scratch Training Epoch {epoch}: Accuracy = {epoch_acc:.2f}%")

                full_scratch_time = time.time() - start_time
                full_scratch_time_list.append(full_scratch_time)
                scratch_acc_history.extend(scratch_iter_accs)

                # if e >= len(Weight_Diff_Record)-10:
                # # Full scratch training with iteration-level evaluation
                #     scratch_iter_accs = []
                #     start_time = time.time()
                #     scratch_model = CNN().to(device)
                #     optimizer = optim.SGD(scratch_model.parameters(), lr=lr)
                #     criterion = nn.CrossEntropyLoss()

                #     for epoch in range(len(old_param_list_r[0:e+9])):
                #         for data, target in train_loader:
                #             data, target = data.to(device), target.to(device)
                #             optimizer.zero_grad()
                #             outputs = scratch_model(data)
                #             loss = criterion(outputs, target)
                #             loss.backward()
                #             optimizer.step()

                #         # Evaluate at each epoch
                #         epoch_acc = evaluate_accuracy(test_loader, scratch_model)
                #         scratch_iter_accs.append(epoch_acc)
                #         print(f"Scratch Training Epoch {epoch}: Accuracy = {epoch_acc:.2f}%")

                #     full_scratch_time = time.time() - start_time
                #     full_scratch_time_list.append(full_scratch_time)
                #     scratch_acc_history.extend(scratch_iter_accs)


                print(f"\nEpoch {e:02d} Comparison:")
                print(f"Global (Poisoned) - Accuracy: {global_accuracy:.4f}")
                #print(f"Recovery - Accuracy: {recovery_accuracy:.4f}, Time: {recovery_time:.2f}s")
                #print(f"Full Scratch - Accuracy: {full_scratch_accuracy:.4f}, Time: {full_scratch_time:.2f}s")


                ter = calculate_ter(net, test_loader, device)
                ter_values.append(ter)
                asr = calculate_asr(net, test_loader, target_label, trigger_fn, device)
                asr_values.append(asr)
                T = len(old_param_list_r[0:e])
                acp = calculate_acp(T, Tr_list)
                acp_values.append(acp)

                flag = 0
                malicious_score = np.zeros((1, len(Cr)))
                counter=counter+1

            elif ((e+1)%10==0 and  flag==0):
                start_time = time.time()
                scratch_model = prev_recovered_model
                optimizer = optim.SGD(scratch_model.parameters(), lr=lr)
                criterion = nn.CrossEntropyLoss()

                for i in range(10):
                    for data, target in train_loader:
                        data, target = data.to(device), target.to(device)
                        optimizer.zero_grad()
                        outputs = scratch_model(data)
                        loss = criterion(outputs, target)
                        loss.backward()
                        optimizer.step()

                    # Evaluate at each epoch
                    epoch_acc = evaluate_accuracy(test_loader, scratch_model)
                    recover_accuracy_list.append(epoch_acc)
                    recovery_acc_history.append(epoch_acc)
                    print(f"recovery Training Epoch {(e-9)+i}: Accuracy = {epoch_acc:.2f}%")
                prev_recovered_model = scratch_model
                flag = 0

            e=e+ 1
            display_detection_metrics(['no', 'gaussian', 'label_flip'], ['K-Means'], attack_metrics)
        # Full scratch training with iteration-level evaluation
        # scratch_iter_accs = []
        # start_time = time.time()
        # scratch_model = CNN().to(device)
        # optimizer = optim.SGD(scratch_model.parameters(), lr=lr)
        # criterion = nn.CrossEntropyLoss()
        # scratch_acc_history=[]
        # for epoch in range(len(old_param_list_r)):
        #     for data, target in train_loader:
        #         data, target = data.to(device), target.to(device)
        #         optimizer.zero_grad()
        #         outputs = scratch_model(data)
        #         loss = criterion(outputs, target)
        #         loss.backward()
        #         optimizer.step()

        #     # Evaluate at each epoch
        #     epoch_acc = evaluate_accuracy(test_loader, scratch_model)
        #     scratch_iter_accs.append(epoch_acc)
        #     print(f"Scratch Training Epoch {epoch}: Accuracy = {epoch_acc:.2f}%")

        # full_scratch_time = time.time() - start_time
        # full_scratch_time_list.append(full_scratch_time)
        # scratch_acc_history.extend(scratch_iter_accs)

        # time_savings = [(full_scratch_time_list[i] - recovery_time_list[i])/full_scratch_time_list[i]*100
        #                 for i in range(len(recovery_time_list)) if recovery_time_list[i] > 0]
        # avg_time_saving = np.mean(time_savings) if time_savings else 0

        print(f"\nFinal Comparison for m={m}:")
        print(f"TER={ter:.2f}%, ASR={asr:.2f}%, ACP={sum(acp_values)/len(acp_values):.2f}%")
        #print(f"Average Time Saving (Recovery vs Full Scratch): {avg_time_saving:.2f}%")
       # print(f"Final Accuracies - Global: {global_acc_list[-1]:.2f}%, Recovery: {recovery_acc_history[-1]:.2f}%, Full Scratch: {full_scratch_acc_list[-1]:.2f}%")
    global_acc_list.insert(0,9.059999465942383)
    recover_accuracy_list.insert(0,9.059999465942383)
    scratch_iter_accs.insert(0,9.059999465942383)

    # Modify the CSV writing to include iteration-level accuracies
    with open('model_comparison_metrics_gauassian_only_Exp_40C_40ep_30start_6_mnist_mudhog_last1_final_1_7.csv', 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow([
            'Epoch', 'Global_Acc', 'Recovery_Acc',
            'Full_Scratch_Acc',

        ])

        for i, epoch in enumerate(range(e+1)):
            writer.writerow([
                epoch ,  # Epoch number (1-based indexing)
                global_acc_list[i],
                recover_accuracy_list[i],
                scratch_iter_accs[i],
            ])



    print("time of training scratch",full_scratch_time_list[-1],"time for recovery",sum(recovery_time_list))
    plt.figure(figsize=(20, 10))

    plt.subplot(2, 2, 1)
    plt.plot(range(len(global_acc_list[0:41])), global_acc_list[0:41], label='Global (Poisoned)', marker='^')

    plt.plot(range(len(scratch_iter_accs)), scratch_iter_accs, label='Full Scratch', marker='x')
    plt.plot(range(len(recover_accuracy_list)), recover_accuracy_list, label='Recovery', marker='o')

    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid()


    plt.subplot(2, 2, 2)
    plt.bar(['Full Scratch','Recovery'], [full_scratch_time_list[-1],sum(recovery_time_list)], color=['red', 'green'])
    # plt.plot(1, full_scratch_time_list[-1], label='Full Scratch', marker='x')
    # plt.plot(1, sum(recovery_time_list), label='Recovery', marker='o')

    plt.xlabel('MNIST')
    plt.ylabel('Time (seconds)')
    plt.legend()
    plt.grid()

    plt.subplot(2, 2, 3)
    #plt.plot(m_values, ter_values, marker='o')
    plt.xlabel('m (attack ratio)')
    plt.ylabel('TER (%)')
    plt.grid()

    plt.subplot(2, 2, 4)
    plt.plot(m_values, asr_values, marker='o')
    plt.xlabel('m (attack ratio)')
    plt.ylabel('ASR (%)')
    plt.grid()

    plt.tight_layout()
    plt.savefig('model_comparison_metrics.png')
    plt.show()

if __name__ == "__main__":
    args = parse_args()
    main(args)
